# E v9 engineering smoke
Private frozen development input; Qwen2.5-7B NF4; 5 NQ + 5 FEVER across three arms.
Maximum 70 calls. This run checks execution and replay, not gain or prompt selection.

In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys, os, tarfile
EXPECTED_CHECKSUMS = "20eb87734301f5947e30ae203ba21b24ea3002c491f784f1f2279630dc878216"
EXPECTED_ARCHIVE = "4f3e3e2acf4de62bf70089b2a5ee13b66afc917641c8f602f6dc6faaa79046bf"
base = Path("/kaggle/input")
def sha(path): return hashlib.sha256(path.read_bytes()).hexdigest()
candidates = [p for p in base.rglob("bundle_checksums.json") if sha(p) == EXPECTED_CHECKSUMS]
if not candidates:
    archives = [p for p in base.rglob("answer_repair_v9_dev_v1_gpu.tar.gz") if sha(p) == EXPECTED_ARCHIVE]
    assert len(archives) == 1, "Attach the private v9 frozen input dataset"
    extract = Path("/kaggle/working/v9_verified_input")
    extract.mkdir(exist_ok=True)
    with tarfile.open(archives[0]) as tar:
        for member in tar.getmembers():
            target = (extract / member.name).resolve()
            assert target.is_relative_to(extract.resolve()) and member.isfile(), "Unsafe archive member"
        tar.extractall(extract)
    candidates = [p for p in extract.rglob("bundle_checksums.json") if sha(p) == EXPECTED_CHECKSUMS]
assert len(candidates) == 1, "Frozen input fingerprint mismatch"
source = candidates[0].parent
checksums = json.loads(candidates[0].read_text())
for name, expected in checksums.items():
    target = (source / name).resolve()
    assert target.is_relative_to(source.resolve()) and sha(target) == expected, name
root = Path("/kaggle/working/e_v9_smoke_" + EXPECTED_CHECKSUMS[:12])
shutil.copytree(source, root, dirs_exist_ok=True)
os.chdir(root)
(root / "bootstrap.json").write_text(json.dumps({"expected_checksums_sha256":EXPECTED_CHECKSUMS,"expected_archive_sha256":EXPECTED_ARCHIVE,"purpose":"10-sample exposed development engineering smoke; no EM-based selection"}, indent=2))
subprocess.run([sys.executable,"-m","pip","install","-r","requirements-answer-repair.txt"],check=True)
subprocess.run(["sh","RUN_SMOKE.sh"],check=True)
outputs = list((root/"results").glob("*.jsonl"))
assert len(outputs) == 6
assert all(len(p.read_text().splitlines()) == 5 for p in outputs)
print("V9 ENGINEERING SMOKE COMPLETE: 30 predictions, 6 replay checks; no effectiveness claim.")
